In [2]:
import torch

In [3]:
! nvidia-smi

Sun Apr 26 17:47:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.21                 Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   40C    P8             17W /  200W |    1287MiB /  12282MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
print(torch.cuda.is_available())
print(f'* CUDA Device: {torch.cuda.get_device_name("cuda:0")}\n* Device Properties: {torch.cuda.get_device_properties("cuda:0")}')

# device = torch.cuda.device(0)
device = torch.device('cuda:0')

True
* CUDA Device: NVIDIA GeForce RTX 4070
* Device Properties: _CudaDeviceProperties(name='NVIDIA GeForce RTX 4070', major=8, minor=9, total_memory=12281MB, multi_processor_count=46, uuid=f2da3a1e-bdf3-094c-8f5b-f221eb8ef48a, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=36MB)


In [5]:
import os
import nibabel as nib
import json
from pathlib import Path

In [16]:
DATASET_DIR = Path('C:/Users/sammi/Desktop/projects/brats-2023/dataset')
CHALLENGES = ["GLI", "MEN", "PED"]

In [23]:
def load_dataset(challenge):
    challenge_dir = DATASET_DIR / f"ASNR-MICCAI-BraTS2023-{challenge}-Challenge-TrainingData"
    dataset = []

    for t1c_file in challenge_dir.rglob('*-t1c.nii.gz'):
        subject_dir = t1c_file.parent
        # subject_name = subject_dir.name

        subject_files = {
            "t1c": subject_dir / f"{subject_dir.name}-t1c.nii.gz",
            "t1n": subject_dir / f"{subject_dir.name}-t1n.nii.gz",
            "t2f": subject_dir / f"{subject_dir.name}-t2f.nii.gz",
            "t2w": subject_dir / f"{subject_dir.name}-t2w.nii.gz",
            "seg": subject_dir / f"{subject_dir.name}-seg.nii.gz"
        }

        # Check if all files exist
        if all(path.exists() for path in subject_files.values()):
            dataset.append(subject_files)
        else:
            # This field is yet to be verified
            print(f"Warning: Missing files for subject {subject_dir.name}, skipping.")

    return dataset

In [24]:
datasets = {challenge: load_dataset(challenge) for challenge in CHALLENGES}

In [27]:
def save_splits(datasets, output_dir="splits"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    for challenge, data in datasets.items():
        split_point = int(0.8 * len(data))
        train_split = data[:split_point]
        val_split = data[split_point:]

        with open(output_dir / f"{challenge}_train.json", "w") as train_file:
            # json.dump([str(item) for item in train_data], f, indent=4)
            json.dump(train_split, train_file, indent=4, default=str)

        with open(output_dir / f"{challenge}_val.json", "w") as val_file:
            # json.dump([str(item) for item in val_data], f, indent=4)
            json.dump(val_split, val_file, indent=4, default=str)

In [28]:
save_splits(datasets)